# The AGUI 13-Event Protocol

**What this notebook shows**: the 13 AGUI event types emitted by the ADK agent tree, the `render_agui_events()` loop, and the SSE stream to `/api/copilotkit/chat/completions`.

**Source**: `gemini_hackathon/agents/adk_gemini_agent.py:AGUI_EVENT_TYPES` + `render_agui_events()`.


## Layer 3 — The 13 AGUI event types

Per `AGUI_EVENT_TYPES` (the gemini-hackathon AG-UI subset):

1. `RUN_STARTED`
2. `STATE_DELTA`
3. `TEXT_MESSAGE_START`
4. `TEXT_MESSAGE_CONTENT`
5. `TEXT_MESSAGE_END`
6. `TOOL_CALL_START`
7. `TOOL_CALL_ARGS`
8. `TOOL_CALL_END`
9. `TOOL_CALL_RESULT`
10. `STEP_STARTED`
11. `STEP_FINISHED`
12. `RUN_FINISHED`
13. `RUN_ERROR`


In [ ]:
from gemini_hackathon.agents.adk_gemini_agent import AGUI_EVENT_TYPES, AgUiEvent

print(f"13 AGUI events ({len(AGUI_EVENT_TYPES)}):")
for i, evt in enumerate(AGUI_EVENT_TYPES, 1):
    print(f"  {i:>2}. {evt}")


## The `render_agui_events()` loop

The 33-LOC loop that converts `google.adk.events.Event` objects into `AgUiEvent` instances:
1. Iterates the ADK Event stream
2. For each event with `author == "agent"`: emits TEXT_MESSAGE_CONTENT + TOOL_CALL_* for function_calls
3. For each event with non-agent author: emits TOOL_CALL_RESULT for function_response
4. Returns a list[AgUiEvent] that the FastAPI handler serialises to SSE


In [ ]:
import inspect
from gemini_hackathon.agents.adk_gemini_agent import render_agui_events
print(inspect.getsource(render_agui_events))


## The SSE stream from `/api/copilotkit/chat/completions`

The route handler in `gemini_hackathon/backend.py:_handle_agents_chat`:
1. Parses the JSON body (message, user_id, session_id, subnation, role, cycle, ...)
2. Calls `run_agent_turn(...)` to get back the `AgentTurnResult`
3. Serialises the `events: list[AgUiEvent]` to SSE chunks
4. The CopilotKit React provider in `web/src/routes/__root.tsx:14` consumes the SSE stream via `useFrontendTool` / `useRenderTool`
